In [ ]:
import pandas as pd
import os


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 0)

In [ ]:
#getting csv with the viral concepts that will be chorts
concepts_csv = '/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/viral_cohort_patient_overlap_seasonal_nonseasonal_update1.csv'
viral_concept_df = pd.read_csv(concepts_csv)



In [ ]:
#nonseasonal W/ vaccination DF

In [ ]:
#mapping specie : concept_ids 
vax_table = viral_concept_df.loc[:, ['concept_id','standard_concept_name','non_seasonal_vax', 'specie', 'genus'] ]
ns_vax_table = vax_table.loc[vax_table['non_seasonal_vax'] == 'Y', :].copy()


def merge_specie_data(new_column):
    
    if new_column["genus"] == "Influenzavirus":
        return new_column["genus"]
    else:
        return new_column["specie"]
    

    
ns_vax_table["updated_specie"] = ns_vax_table.apply(merge_specie_data, axis=1)
ns_table = ns_vax_table.loc[:, ['concept_id','standard_concept_name', 'updated_specie']]
ns_table

In [ ]:
#mapping specie : concept_ids 
vax_table = viral_concept_df.loc[:, ['concept_id','standard_concept_name','non_seasonal_vax', 'specie', 'genus'] ]
ns_vax_table = vax_table.loc[vax_table['non_seasonal_vax'] == 'Y', :].copy()


def merge_specie_data(new_column):
    
    if new_column["genus"] == "Influenzavirus":
        return new_column["genus"]
    else:
        return new_column["specie"]
    

    
ns_vax_table["updated_specie"] = ns_vax_table.apply(merge_specie_data, axis=1)
ns_table = ns_vax_table.loc[:, ['concept_id','standard_concept_name', 'updated_specie']]
ns_table

In [ ]:
ns_vax_table['concept_id'].nunique()

In [ ]:
#mapping specie:vaccine list
concepts_csv1 = '/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/ns_viral_cohort_vax_ids.csv'
vaccine_concepts = pd.read_csv(concepts_csv1)
vaccine_concept_df = vaccine_concepts.loc[:, ['updated_specie', 'vaccine_id']]
vaccine_concept_df

In [ ]:
ns_cohort_vaccine_df = ns_table.merge(vaccine_concept_df,  on='updated_specie', how='left')
ns_cohort_vaccine_df

In [ ]:
def get_concept_vax_id_dict(ns_cohort_vax_df): 
    '''
       1. subset a new df with only 2 columns: concept_id and vaccine_id
       2. reset the index to concept_id
       3. groupby concept_id, selecting (vaccine_id) col as a series, and using .agg to apply an aggregation to each 
        group, that being the lambda function to make the grouped series into a single list
    
    ''' 
    
    df1 = ns_cohort_vax_df.loc[ :, ['concept_id', 'vaccine_id']]
    df2 = df1.set_index('concept_id')
    df3 = df2.groupby('concept_id', sort=False)['vaccine_id'].agg(lambda vax_id: list(vax_id))
    final = df3.to_dict()
    
    
    return final

In [ ]:
cohort_vax_dict = get_concept_vax_id_dict(ns_cohort_vaccine_df)
cohort_vax_dict

In [ ]:
def get_condition_summary(concept_id):
    """
    Fetches per-patient summary for the specified condition_concept_id,
    applying Cohort Builder UI filters (EHR + genomics, observation window,
    flat-events, standard concepts), and returns a DataFrame with one row per
    patient including:
      - condition_concept_id
      - standard_concept_name, standard_vocabulary
      - first and last diagnosis date for that concept
      - condition_type_concept_name, visit_occurrence_concept_name from first occurrence
      - visits_for_concept: count of unique visit_occurrence_id for the concept
      - visits_all_concepts: count of unique visits across all conditions
      - concept_count_ehr: count of distinct condition concepts in EHR
    """
    dataset = os.environ["WORKSPACE_CDR"]
    sql = f"""
    WITH
      ehr_genomics_patients AS (
        SELECT DISTINCT person_id
        FROM `{dataset}.cb_search_person`
        WHERE has_ehr_data = 1
          AND (
               has_whole_genome_variant      = 1
            OR has_lr_whole_genome_variant   = 1
            OR has_array_data                = 1
          )
      ),

      all_occ AS (
        SELECT
          co.person_id,
          co.condition_concept_id,
          co.visit_occurrence_id,
          co.condition_start_datetime,
          co.condition_end_datetime,
          co.condition_type_concept_id
        FROM `{dataset}.condition_occurrence` co
        JOIN ehr_genomics_patients eg
          ON co.person_id = eg.person_id

        -- only events that made it into the CB search table
        JOIN `{dataset}.cb_search_all_events` ev
          ON ev.person_id = co.person_id
         AND ev.concept_id = co.condition_concept_id
         AND DATE(co.condition_start_datetime) = ev.entry_date

        JOIN `{dataset}.concept` c_std
          ON co.condition_concept_id = c_std.concept_id
        WHERE c_std.standard_concept = 'S'
      ),

      spec_occ AS (
        SELECT *
        FROM all_occ
        WHERE condition_concept_id = {concept_id}
      ),

      detail AS (
        SELECT
          person_id,
          condition_concept_id,
          condition_start_datetime AS first_diag_date,
          condition_end_datetime   AS first_end_date,
          condition_type_concept_id,
          visit_occurrence_id,
          ROW_NUMBER() OVER (PARTITION BY person_id ORDER BY condition_start_datetime) AS rn
        FROM spec_occ
      ),

      first_detail AS (
        SELECT
          d.person_id,
          d.condition_concept_id,
          d.first_diag_date,
          d.first_end_date,
          c_std.concept_name        AS standard_concept_name,
          c_std.vocabulary_id       AS standard_vocabulary,
          c_type.concept_name       AS condition_type_concept_name,
          vis_evt.concept_name      AS visit_occurrence_concept_name
        FROM detail d
        JOIN `{dataset}.concept` c_std
          ON d.condition_concept_id = c_std.concept_id
        LEFT JOIN `{dataset}.concept` c_type
          ON d.condition_type_concept_id = c_type.concept_id
        LEFT JOIN `{dataset}.visit_occurrence` v
          ON d.visit_occurrence_id = v.visit_occurrence_id
        LEFT JOIN `{dataset}.concept` vis_evt
          ON v.visit_concept_id = vis_evt.concept_id
        WHERE d.rn = 1
      ),

      spec_metrics AS (
        SELECT
          person_id,
          MIN(condition_start_datetime) AS first_diag_date,
          MAX(condition_start_datetime) AS last_diag_date,
          COUNT(DISTINCT visit_occurrence_id) AS visits_for_concept
        FROM spec_occ
        GROUP BY person_id
      ),

      allv AS (
        SELECT
          person_id,
          COUNT(DISTINCT visit_occurrence_id) AS visits_all_concepts
        FROM all_occ
        GROUP BY person_id
      ),

      conc AS (
        SELECT
          person_id,
          COUNT(DISTINCT condition_concept_id) AS concept_count_ehr
        FROM all_occ
        GROUP BY person_id
      )

    SELECT
      fd.person_id,
      fd.condition_concept_id,
      fd.standard_concept_name,
      fd.standard_vocabulary,
      fd.first_diag_date,
      sm.last_diag_date,
      fd.condition_type_concept_name,
      fd.visit_occurrence_concept_name,
      sm.visits_for_concept,
      av.visits_all_concepts,
      cc.concept_count_ehr
    FROM first_detail fd
    JOIN spec_metrics sm  ON fd.person_id = sm.person_id
    LEFT JOIN allv av       ON fd.person_id = av.person_id
    LEFT JOIN conc cc       ON fd.person_id = cc.person_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df

In [ ]:
def get_vaccine_table(vaccine_ids):
    """
    Fetches drug exposure rows for the specified vaccine_concept_id(s),
    applying Cohort Builder UI filters (EHR + genomics, flat‐events,
    standard concepts), and returns a DataFrame with:
      - person_id
      - drug_concept_id
      - standard_concept_name
      - drug_exposure_start/end_datetime
      - verbatim_end_date
      - source_concept_name
    """
    dataset = os.environ["WORKSPACE_CDR"]
    # allow passing either a single int or a list/tuple of ints
    if not isinstance(vaccine_ids, (list, tuple)):
        vaccine_ids = [vaccine_ids]
    ids_sql = "(" + ",".join(str(i) for i in vaccine_ids) + ")"

    sql = f"""
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name AS drug_standard_concept_name,
        d_exposure.drug_exposure_start_datetime,
        d_exposure.drug_exposure_end_datetime,
        d_exposure.verbatim_end_date,
        d_source_concept.concept_name AS source_concept_name 
    FROM (
        SELECT * 
        FROM `{dataset}.drug_exposure` d_exposure 
        WHERE
            drug_concept_id IN (
                SELECT DISTINCT ca.descendant_id 
                FROM `{dataset}.cb_criteria_ancestor` ca 
                JOIN (
                    SELECT DISTINCT c.concept_id       
                    FROM `{dataset}.cb_criteria` c       
                    JOIN (
                        SELECT CAST(cr.id AS STRING) AS id             
                        FROM `{dataset}.cb_criteria` cr             
                        WHERE
                            cr.concept_id IN {ids_sql}           
                            AND cr.full_text LIKE '%_rank1]%'       
                    ) a 
                      ON (
                        c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id
                      ) 
                    WHERE
                        c.is_standard   = 1 
                        AND c.is_selectable = 1
                ) b 
                  ON ca.ancestor_id = b.concept_id
            )
          AND d_exposure.person_id IN (
            SELECT DISTINCT p.person_id  
            FROM `{dataset}.cb_search_person` p  
            WHERE
                p.has_ehr_data = 1 
              AND (
                   p.has_whole_genome_variant    = 1 
                OR p.has_lr_whole_genome_variant = 1 
                OR p.has_array_data              = 1 
              )
          )
          AND d_exposure.person_id IN (
            SELECT criteria.person_id 
            FROM (
              SELECT DISTINCT person_id, entry_date, concept_id 
              FROM `{dataset}.cb_search_all_events` 
              WHERE
                concept_id IN (
                  SELECT DISTINCT c.concept_id 
                  FROM `{dataset}.cb_criteria` c 
                  JOIN (
                    SELECT CAST(cr.id AS STRING) AS id       
                    FROM `{dataset}.cb_criteria` cr       
                    WHERE
                        cr.concept_id IN (440029)       
                      AND cr.full_text LIKE '%_rank1]%'      
                  ) a 
                    ON (
                      c.path LIKE CONCAT('%.', a.id, '.%') 
                      OR c.path LIKE CONCAT('%.', a.id) 
                      OR c.path LIKE CONCAT(a.id, '.%') 
                      OR c.path = a.id
                    ) 
                  WHERE
                    c.is_standard   = 1 
                    AND c.is_selectable = 1
                )
                AND is_standard = 1
            ) criteria
          )
    ) d_exposure 
    LEFT JOIN `{dataset}.concept` d_standard_concept 
      ON d_exposure.drug_concept_id       = d_standard_concept.concept_id 
    LEFT JOIN `{dataset}.concept` d_source_concept 
      ON d_exposure.drug_source_concept_id = d_source_concept.concept_id
    """

    df = pd.read_gbq(
        sql,
        project_id=os.environ.get("BIGQUERY_PROJECT"),
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook",
    )
    return df


In [ ]:
def demographics_table():
    """
    Fetch person demographics rows for a single concept_id,
    using your original SQL structure and injecting concept_id directly.
    """
    dataset = os.environ["WORKSPACE_CDR"]

    demographics_sql  = f"""
    SELECT
        person.person_id,
        
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
   
        p_race_concept.concept_name as race,
    
        p_ethnicity_concept.concept_name as ethnicity,
    
        p_sex_at_birth_concept.concept_name as sex_at_birth,
      
    FROM
        `{dataset}.person` person 
    LEFT JOIN
        `{dataset}.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `{dataset}.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `{dataset}.cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `{dataset}.cb_search_person` p 
            WHERE
                has_array_data = 1 ) )"""


    demographics_df = pd.read_gbq(
        demographics_sql,
        dialect="standard",
        use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
        progress_bar_type="tqdm_notebook"
    )

    return demographics_df

In [ ]:
COND_NAME_MAP = dict(
    zip(ns_table['concept_id'], ns_table['standard_concept_name']))


def merge_cond_vax_table(ns_vax_id_dict): 
    
    ns_vax_cohort_dict = {}
    
    demo = demographics_table().drop_duplicates('person_id')
    
    
    for key, vax_list in ns_vax_id_dict.items() :
    
        cond = get_condition_summary(key)
        vax = get_vaccine_table(vax_list)
        
        cohort_base = cond.merge(demo,  on='person_id', how='left')
        
        vax_cohort =cond.merge(vax , on='person_id', how='inner')
        
        pre_vax_df = vax_cohort.loc[vax_cohort['drug_exposure_start_datetime']  < vax_cohort['first_diag_date']].copy()
        
        pre_vax_df = (
            pre_vax_df
            .sort_values("drug_exposure_start_datetime")
            .drop_duplicates(subset="person_id", keep="first"))
        
        
        final_merge = cohort_base.merge(pre_vax_df, on = 'person_id', how = 'left' )
        
        #find dictionary keys, and add final tables to dict
        cond_name = COND_NAME_MAP.get(key, "<unknown>")
        ns_vax_cohort_dict[(key, cond_name)] = final_merge
        
        
       

    return ns_vax_cohort_dict

In [ ]:
ns_vax_df = merge_cond_vax_table(cohort_vax_dict)

In [ ]:
def merge_race_ethnicity_data(new_column):
    
    if new_column["ethnicity"] == "Hispanic or Latino":
        return new_column["ethnicity"]
    else:
        return new_column["race"]
    
def add_vax_count_by_ethnicity(new_column):
    
    if new_column["drug_exposure_start_datetime"] is pd.NaT:
        return 'N'
    else:
        return 'Y'
    
    if new_column["ethnicity"] == "Hispanic or Latino":
        return new_column["ethnicity"]
    else:
        return new_column["race"]
  



In [ ]:
for key, table in ns_vax_df.items():
    
    table["updated_race"] = table.apply(merge_race_ethnicity_data, axis=1)
    table["vaccinated"] = table.apply(add_vax_count_by_ethnicity, axis=1)

In [ ]:
ns_vax_df

In [ ]:

import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_cohort_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(ns_vax_df, f)

print(f"Saved {len(ns_vax_df)} DataFrames to {out_file}")




In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_cohort_data_table.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    ns_vax_df = pickle.load(f)

print("Reloaded keys:", list(ns_vax_df.keys()))

In [ ]:
ns_vax_df

In [ ]:
def get_ns_ethnicity_vax_binning(ns_vax_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in ns_vax_df.items():
        
        
        grouped = table.groupby('updated_race')[['vaccinated']].value_counts()
     

        df1 = pd.DataFrame(grouped)
        df1['concept_id'] = key[0] #make a concept_id column with key values
        df1['concept_name'] = key[1]
        df2 = df1.reset_index()
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['concept_id', 'concept_name','updated_race', 'vaccinated', 'count']
    ns_table = ns_table[new_order]
    final = ns_table.rename(columns={'concept_id': 'condition_concept_id_x', 'concept_name': 'standard_concept_name_x'})
    
        
        
    return final

ns_ethnicity_vax_race_bin_count = get_ns_ethnicity_vax_binning(ns_vax_df)
ns_ethnicity_vax_race_bin_count

In [ ]:
# 1) List of races to exclude
to_drop = [
    'PMI: Skip',
    'None of these',
    'American Indian or Alaska Native',
    'I prefer not to answer',
    "more than one population" 
]

# 2) Keep only rows whose updated_race is not in that list
df_filtered = ns_ethnicity_vax_race_bin_count[~ns_ethnicity_vax_race_bin_count['updated_race'].isin(to_drop)]

In [ ]:
df_filtered
df_filtered

In [ ]:
#FILTERING

In [ ]:
import pandas as pd
import numpy as np


# -----------------------------------------------------------------------------
# 1) Build per-(concept, race, vax) counts (no pre-filtering)
# -----------------------------------------------------------------------------
race_vax = (
    df_filtered
      .groupby(
          ['standard_concept_name_x', 'updated_race', 'vaccinated'],
          as_index=False
      )['count']
      .sum()
      .pivot_table(
          index=['standard_concept_name_x', 'updated_race'],
          columns='vaccinated',
          values='count',
          fill_value=0
      )
      .reset_index()
      .rename(columns={'N': 'N', 'Y': 'Y'})
)

# compute total patients per race
race_vax['race_total'] = race_vax['Y'] + race_vax['N']

# -----------------------------------------------------------------------------
# 2) Collapse to one row per concept with summary features
# -----------------------------------------------------------------------------
concept_summary = (
    race_vax
      .groupby('standard_concept_name_x', as_index=False)
      .agg(
        total_races     = ('updated_race', 'nunique'),                  # number of distinct races
        complete_races  = ('race_total',  lambda x: ((race_vax.loc[x.index,'Y'] > 100) &
                                                     (race_vax.loc[x.index,'N'] > 100)).sum()),
        eligible_races  = ('race_total',  lambda x: (x >= 100).sum())    # races with total ≥100
      )
)

# -----------------------------------------------------------------------------
# 3) Define cohorts C1–D3
# -----------------------------------------------------------------------------
conds = [
    # C1: ≥2 races each with Y>100 & N>100
    concept_summary['complete_races'] >= 2,

    # C2: exactly 1 race with Y>100 & N>100
    concept_summary['complete_races'] == 1,

    # D1: no complete races, but ≥2 races with total ≥100
    (concept_summary['complete_races'] == 0) &
    (concept_summary['eligible_races']  >= 2),

    # D2: no complete races, exactly 1 race with total ≥100
    (concept_summary['complete_races'] == 0) &
    (concept_summary['eligible_races']  == 1),
    
]
choices = ['C1', 'C2', 'D1', 'D2']

# anything else → D3
concept_summary['cohort_group'] = np.select(conds, choices, default='D3')

# -----------------------------------------------------------------------------
# 4) (Optional) Merge cohort_group back onto the raw DataFrame
# -----------------------------------------------------------------------------
df_all = (
    df_filtered
      .merge(
          concept_summary[['standard_concept_name_x', 'cohort_group']],
          on='standard_concept_name_x',
          how='left'
      )
)

# -----------------------------------------------------------------------------
# 5) Sanity checks
# -----------------------------------------------------------------------------
print("Concepts per cohort:")
print(concept_summary['cohort_group'].value_counts(), "\n")

print("Examples by cohort:")
for grp in ['C1','C2','D1','D2','D3']:
    ex = concept_summary.loc[
        concept_summary['cohort_group'] == grp, 'standard_concept_name_x'
    ].unique()[:]
    print(f" {grp}: {ex}")


In [ ]:
c1  = concept_summary.loc[concept_summary['cohort_group'] == 'C1', :].copy()
c1

In [ ]:
c2  = concept_summary.loc[concept_summary['cohort_group'] == 'C2', :].copy()
c2

In [ ]:
d1  = concept_summary.loc[concept_summary['cohort_group'] == 'D1', :].copy()
d1

In [ ]:
d1['eligible_races'].sum()

In [ ]:
d2  = concept_summary.loc[concept_summary['cohort_group'] == 'D2', :].copy()
d2

In [ ]:
d3  = concept_summary.loc[concept_summary['cohort_group'] == 'D3', :].copy()
d3

In [ ]:
#getting person_Id from filtering

In [ ]:
#C1 and C2

In [ ]:
import pandas as pd
import numpy as np

# ─── INPUTS ────────────────────────────────────────────────────────────────────
# 1) ns_vax_df: dict mapping
#      (condition_concept_id, standard_concept_name)
#    → full person-level DataFrame with at least:
#      ['person_id', 'updated_race', 'vaccinated', …]
#
# 2) df_all: DataFrame with columns
#      ['condition_concept_id_x',
#       'standard_concept_name_x',
#       'updated_race',
#       'vaccinated',
#       'count',
#       'cohort_group']
#    as produced in your step (4). We’ll re-derive race-level completeness from its counts.

# ─── 1) Compute per-(concept, race) N/Y counts & mark “complete” races ─────────
race_counts = (
    df_all
    .pivot_table(
        index=['condition_concept_id_x', 'standard_concept_name_x', 'updated_race'],
        columns='vaccinated',
        values='count',
        aggfunc='sum',
        fill_value=0
    )
    .reset_index()
    .rename(columns={'N': 'N_count', 'Y': 'Y_count'})
)
race_counts['complete_race'] = (
    (race_counts['N_count'] > 100) &
    (race_counts['Y_count'] > 100)
)

# ─── 2) Summarize per-concept how many “complete” races there are & assign C1/C2 ─
concept_summary = (
    race_counts
    .groupby(
        ['condition_concept_id_x', 'standard_concept_name_x'],
        as_index=False
    )
    .agg(complete_races=('complete_race', 'sum'))
)
conds = [
    concept_summary['complete_races'] >= 2,   # C1
    concept_summary['complete_races'] == 1,   # C2
]
choices = ['C1', 'C2']
concept_summary['cohort_group'] = np.select(conds, choices, default=None)

# keep only the C1/C2 concepts
concept_summary = concept_summary[
    concept_summary['cohort_group'].isin(['C1','C2'])
].copy()

# ─── 3) Filter race_counts to only those complete races of C1/C2 concepts ──────
race_complete = (
    race_counts
    .merge(
        concept_summary,
        on=['condition_concept_id_x', 'standard_concept_name_x'],
        how='inner'
    )
)
race_complete = race_complete[race_complete['complete_race']].copy()

# ─── 4) Build your dict of person-level DataFrames for each slice ──────────────
# Key = (concept_id, concept_name, race, vaccinated_flag, cohort_group)
vax_person_dfs = {}
for row in race_complete.itertuples(index=False):
    cid   = row.condition_concept_id_x
    name  = row.standard_concept_name_x
    race  = row.updated_race
    cohort = row.cohort_group

    # grab full person-level DF for this concept
    df_full = ns_vax_df[(cid, name)]

    # split out N and Y for this race
    for flag in ['N','Y']:
        df_slice = df_full[
            (df_full['updated_race'] == race) &
            (df_full['vaccinated']     == flag)
        ].copy()

        key = (cid, name, race, flag, cohort)
        vax_person_dfs[key] = df_slice




In [ ]:
vax_person_dfs.keys()

In [ ]:
len(vax_person_dfs.keys())

In [ ]:
import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_C1_C2_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(vax_person_dfs, f)

print(f"Saved {len(vax_person_dfs)} DataFrames to {out_file}")

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_C1_C2_data_table.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    c1c2 = pickle.load(f)



In [ ]:
for keys in c1c2.keys():
    print(keys)

In [ ]:
#D1 and D2 

In [ ]:
import pandas as pd
import numpy as np

# ─── INPUTS ────────────────────────────────────────────────────────────────────
# 1) ns_vax_df: dict mapping
#      (condition_concept_id, standard_concept_name)
#    → full person-level DataFrame, with columns at least:
#      ['person_id', 'updated_race', 'vaccinated', …]
#
# 2) df_all: DataFrame with columns
#      ['condition_concept_id_x',
#       'standard_concept_name_x',
#       'updated_race',
#       'vaccinated',
#       'count']
#    (i.e. the per-race, per-vax counts you computed earlier)

# ─── 1) Compute per-(concept, race) totals & mark “complete” races ────────────
race_counts = (
    df_all
      .pivot_table(
          index=['condition_concept_id_x', 'standard_concept_name_x', 'updated_race'],
          columns='vaccinated',
          values='count',
          aggfunc='sum',
          fill_value=0
      )
      .reset_index()
)
race_counts.columns.name = None
race_counts = race_counts.rename(columns={'N': 'N_count', 'Y': 'Y_count'})
race_counts['race_total']    = race_counts['N_count'] + race_counts['Y_count']
race_counts['complete_race'] = (
    (race_counts['N_count'] > 100) &
    (race_counts['Y_count'] > 100)
)

# ─── 2) Summarize per-concept → how many complete vs. eligible races ──────────
concept_summary = (
    race_counts
      .groupby(
          ['condition_concept_id_x', 'standard_concept_name_x'],
          as_index=False
      )
      .agg(
        complete_races = ('complete_race', 'sum'),
        eligible_races = ('race_total', lambda x: (x >= 100).sum())
      )
)

# assign D1 / D2
conds = [
    (concept_summary['complete_races'] == 0) &
      (concept_summary['eligible_races'] >= 2),  # D1
    (concept_summary['complete_races'] == 0) &
      (concept_summary['eligible_races'] == 1)   # D2
]
choices = ['D1','D2']
concept_summary['cohort_group'] = np.select(conds, choices, default=None)

# keep only D1/D2 concepts
d_concepts = concept_summary[
    concept_summary['cohort_group'].isin(['D1','D2'])
].copy()

# ─── 3) For those concepts, pick only the eligible races (race_total ≥100) ────
d_races = (
    race_counts
      .merge(
          d_concepts,
          on=['condition_concept_id_x','standard_concept_name_x'],
          how='inner'
      )
)
d_races = d_races[d_races['race_total'] >= 100].copy()

# ─── 4) Build dict of person-level DFs for each (concept, race, cohort) ───────
# Key = (condition_concept_id, concept_name, updated_race, cohort_group)
person_dfs = {}
for row in d_races.itertuples(index=False):
    cid    = row.condition_concept_id_x
    name   = row.standard_concept_name_x
    race   = row.updated_race
    cohort = row.cohort_group

    # original full DF for this concept
    df_full = ns_vax_df[(cid, name)]

    # keep ONLY that race (both vaccinated & unvaccinated)
    df_slice = df_full[df_full['updated_race'] == race].copy()

    person_dfs[(cid, name, race, cohort)] = df_slice


In [ ]:
person_dfs.keys()

In [ ]:
import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_D1_D2_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(person_dfs, f)

print(f"Saved {len(person_dfs)} DataFrames to {out_file}")

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_D1_D2_data_table.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    d1d2 = pickle.load(f)

In [ ]:
for keys in d1d2.keys():
    print(keys)

In [ ]:
#raw Vaccinated Y / N cocnepts

In [ ]:
def get_ns_vax_binning(ns_vax_df):
    
    ns_cohort_bin_dict = {}
    
    for key, table in ns_vax_df.items():
        
        
        grouped = table.groupby('standard_concept_name_x')[['vaccinated']].value_counts()
     

        df1 = pd.DataFrame(grouped)
        df1['concept_id'] = key[0] #make a concept_id column with key values
        df1['concept_name'] = key[1]
        df2 = df1.reset_index()
        
        ns_cohort_bin_dict[key] = df2
        
        
        
    ns_table = pd.concat(ns_cohort_bin_dict.values(), axis = 0, ignore_index = True)
    new_order = ['concept_id', 'concept_name', 'vaccinated', 'count']
    ns_table = ns_table[new_order]
    final = ns_table.rename(columns={'concept_id': 'condition_concept_id_x', 'concept_name': 'standard_concept_name_x'})
   
        
        
    return final

ns_vax_race_bin_count = get_ns_vax_binning(ns_vax_df)
ns_vax_race_bin_count

In [ ]:
vax_df = (
    ns_vax_race_bin_count
      .groupby(
          ['standard_concept_name_x', 'vaccinated'],
          as_index=False
      )['count']
      .sum()
      .pivot_table(
          index=['standard_concept_name_x'],
          columns='vaccinated',
          values='count',
          fill_value=0
      )
      .reset_index()
      .rename(columns={'N': 'N', 'Y': 'Y'})
)

# compute total patients per race
vax_df['vax_total'] = vax_df['Y'] + vax_df['N']

#removing rows where Y and N arent >= 100
vax_cohort = vax_df.loc[(vax_df['Y'] >= 100) & (vax_df['N'] >= 100), :]
vax_cohort                   

In [ ]:
vax_cohort.shape

In [ ]:
#getting person_Id from filtering

In [ ]:
import pandas as pd

# ─── 0) INPUTS ─────────────────────────────────────────────────────────────────
# ns_vax_df: dict mapping (concept_id, concept_name) → person-level DataFrame
#   each DF must have at least these columns:
#     ['person_id', 'vaccinated', …]
#
# vax_cohort: DataFrame with columns
#   ['condition_concept_id', 'standard_concept_name', 'N', 'Y', 'vax_total']
#   listing only the concepts where both N ≥ 100 and Y ≥ 100.
#
# If your vax_cohort only has 'standard_concept_name' (no concept_id),
# you should re-compute it including the id:

# Flatten and recompute if needed:
master_vax = pd.concat(
    [
        df.assign(
            condition_concept_id=key[0],
            standard_concept_name=key[1]
        )
        for key, df in ns_vax_df.items()
    ],
    ignore_index=True
)

# Recompute counts by both id+name+flag:
vax_counts = (
    master_vax
    .groupby(
        ['condition_concept_id_x', 'standard_concept_name_x', 'vaccinated'],
        as_index=False
    )['person_id']
    .count()
    .rename(columns={'person_id': 'count'})
    .pivot_table(
        index=['condition_concept_id_x', 'standard_concept_name_x'],
        columns='vaccinated',
        values='count',
        fill_value=0
    )
    .reset_index()
    .rename(columns={'N': 'N_count', 'Y': 'Y_count'})
)
vax_counts['vax_total'] = vax_counts['N_count'] + vax_counts['Y_count']

# Filter to your ≥100 threshold:
vax_cohort = vax_counts.loc[
    (vax_counts['N_count'] >= 100) &
    (vax_counts['Y_count'] >= 100),
    ['condition_concept_id_x', 'standard_concept_name_x']
]

# ─── 1) Build the triple-key dict ───────────────────────────────────────────────
# Keys = (condition_concept_id, standard_concept_name, vaccinated_flag)
# Values = DataFrame of all person-level rows for that slice
cohort_set = set(
    zip(
        vax_cohort['condition_concept_id_x'],
        vax_cohort['standard_concept_name_x']
    )
)

vax_person_dfs1 = {}
for (cid, name) in cohort_set:
    df_full = ns_vax_df[(cid, name)]
    for flag in ['N', 'Y']:
        df_slice = df_full[df_full['vaccinated'] == flag].copy()
        # keep whatever columns you need—e.g. person_id, plus any others
        vax_person_dfs1[(cid, name, flag)] = df_slice

# ─── USAGE ─────────────────────────────────────────────────────────────────────
# What keys do we have?
print("Available slices:", list(vax_person_dfs1.keys()))

print(len(list(vax_person_dfs1.keys())))

In [ ]:
import pickle
from pathlib import Path

# 1) Suppose `ns_vax_df` is your dict of DataFrames
#    Example: ns_vax_df = {'a': df1, 'b': df2, ...}

# 2) Choose a workspace folder for persistence
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_raw_Y_N_data_table.pkl')

# 3) Save the entire dict in one go
with open(out_file, 'wb') as f:
    pickle.dump(vax_person_dfs1, f)

print(f"Saved {len(vax_person_dfs1)} DataFrames to {out_file}")

In [ ]:
import pandas as pd
import pickle
from pathlib import Path

# …later, in any notebook in the same workspace…
out_file = Path('/home/jupyter/workspaces/duplicateofcreatinginfectiousdiseasecohorts/nonseasonal_vaccinated_raw_Y_N_data_table.pkl')

# 4) Reload with one line:
with open(out_file, 'rb') as f:
    yn = pickle.load(f)

In [ ]:
for keys in yn.keys():
    print(keys)